# Tahap 1: Membangun Data Split untuk Collaborative Filtering

Notebook ini menangani masalah pada data *train/test split* bawaan (dari paper asli). Pada split bawaan, data test memisahkan resep berdasarkan waktu rilisnya, sehingga model tidak akan pernah melihat resep test tersebut di data log training (*cold start* ekstrim).

Di sini, kita merapikan dan membuat Leave-One-Out (LOO) temporal split langsung dari `RAW_interactions.csv` agar Collaborative Filtering bisa mempelajarinya.

In [1]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

# Konfigurasi Path
# Asumsi dijalankan dari dalam folder `nutrition-aware-recipe-recommendation-system/cf/`
FOOD_COM_DIR = Path("../../food.com")
OUTPUT_DIR = Path("outputs/cf_split")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 3

## 1. Memuat Data & Menghapus Rating 0
Rating 0 pada dataset ini ditandai sebagai data yang ambigu, di mana pengguna tidak memberikan sinyal valid kesukaan suatu item.

In [2]:
print("Memuat RAW_interactions.csv...")
raw = pd.read_csv(FOOD_COM_DIR / "RAW_interactions.csv")
df = raw[raw["rating"] > 0].copy()
df["date"] = pd.to_datetime(df["date"])

print(f"Total baris awal (rating > 0): {len(df):,}")

Memuat RAW_interactions.csv...
Total baris awal (rating > 0): 1,071,520


## 2. Interative Co-filtering
Mengurangi tingkat sparsity ekstrem pada matriks dengan membuang *user* atau *item* yang amat jarang berinteraksi.

In [3]:
print(f"Proses filtering (min_user={MIN_USER_INTERACTIONS}, min_item={MIN_ITEM_INTERACTIONS})...")

prev_len = len(df) + 1
iteration = 0

while len(df) < prev_len:
    prev_len = len(df)
    iteration += 1

    # Filter iteratif
    u_counts = df.groupby("user_id").size()
    df = df[df["user_id"].isin(u_counts[u_counts >= MIN_USER_INTERACTIONS].index)]

    i_counts = df.groupby("recipe_id").size()
    df = df[df["recipe_id"].isin(i_counts[i_counts >= MIN_ITEM_INTERACTIONS].index)]

    print(f"  Tahap {iteration}: {df['user_id'].nunique():,} users, {df['recipe_id'].nunique():,} items, {len(df):,} baris dataset")

print("Konvergen!")

Proses filtering (min_user=5, min_item=3)...
  Tahap 1: 22,002 users, 76,187 items, 673,210 baris dataset
  Tahap 2: 19,188 users, 75,049 items, 661,298 baris dataset
  Tahap 3: 19,125 users, 75,018 items, 660,986 baris dataset
  Tahap 4: 19,123 users, 75,017 items, 660,976 baris dataset
  Tahap 5: 19,123 users, 75,017 items, 660,976 baris dataset
Konvergen!


## 3. Membangun Leave-One-Out (LOO) Temporal Split
- Interaksi terakhir = Data *Test*
- Interaksi kedua terakhir = Data *Validation*
- Sisa log interaksi awal = Data *Training*

In [4]:
df = df.sort_values(["user_id", "date", "recipe_id"]).reset_index(drop=True)

train_parts, val_parts, test_parts = [], [], []
skipped = 0

for uid, group in df.groupby("user_id", sort=False):
    if len(group) < 3:
        skipped += 1
        continue
    train_parts.append(group.iloc[:-2])    # Ambil elemen hingga sisa 2 terakhir
    val_parts.append(group.iloc[-2:-1])    # Ambil kedua paling terakhir
    test_parts.append(group.iloc[-1:])     # Ambil satu paling terakhir

train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts,   ignore_index=True)
test_df  = pd.concat(test_parts,  ignore_index=True)

if skipped:
    print(f"Pengguna dengan < 3 interaksi dilewati: {skipped} user.")

## 4. Map ke Indeks Integer
Algoritma Collaborative Filtering seperti SVD/ALS/BPR membutuhkan index *array* terpisah yang diawali dari `0`.

In [5]:
all_users = sorted(df["user_id"].unique())
all_items = sorted(df["recipe_id"].unique())

user2idx = {uid: idx for idx, uid in enumerate(all_users)}
item2idx = {iid: idx for idx, iid in enumerate(all_items)}

for split_df in [train_df, val_df, test_df]:
    split_df["u"] = split_df["user_id"].map(user2idx)
    split_df["i"] = split_df["recipe_id"].map(item2idx)

n_users = len(user2idx)
n_items = len(item2idx)
sparsity = 1 - len(train_df) / (n_users * n_items)

print(f"Data Split Selesai Dibuat!")
print(f"Jumlah User: {n_users:,}")
print(f"Jumlah Item: {n_items:,}")
print(f"Training: {len(train_df):,} interaksi (Sparsity: {sparsity*100:.3f}%)")
print(f"Validation & Test mendapatkan 1 observasi setiap {len(val_df):,} user.")

Data Split Selesai Dibuat!
Jumlah User: 19,123
Jumlah Item: 75,017
Training: 622,730 interaksi (Sparsity: 99.957%)
Validation & Test mendapatkan 1 observasi setiap 19,123 user.


## 5. Simpan Hasil Pemotongan
Menyimpan kembali dictionary maupun pembagian CSV untuk persiapan *training*.

In [6]:
train_df.to_csv(OUTPUT_DIR / "cf_train.csv", index=False)
val_df.to_csv(OUTPUT_DIR  / "cf_val.csv",   index=False)
test_df.to_csv(OUTPUT_DIR / "cf_test.csv",  index=False)

with open(OUTPUT_DIR / "user2idx.pkl", "wb") as f:
    pickle.dump(user2idx, f)
with open(OUTPUT_DIR / "item2idx.pkl", "wb") as f:
    pickle.dump(item2idx, f)

idx2user = {v: k for k, v in user2idx.items()}
idx2item = {v: k for k, v in item2idx.items()}
with open(OUTPUT_DIR / "idx2user.pkl", "wb") as f:
    pickle.dump(idx2user, f)
with open(OUTPUT_DIR / "idx2item.pkl", "wb") as f:
    pickle.dump(idx2item, f)

print(f"Hasil output tersimpan pada direktori: {OUTPUT_DIR}")

Hasil output tersimpan pada direktori: outputs\cf_split
